# KinoCompass – Mult-VAE на GPU

Mult-VAE (Liang et al., 2018) – вариационный автокодировщик: набор оценённых человеком фильмов сжимается в распределение «вкуса», декодировщик возвращает баллы всем ~21 тыс. фильмов-кандидатов. Сетка: размер «вкуса» 200 и 400, верхняя граница β 0,2 и 0,5, по 40 эпох обучения.

Ноутбук клонирует репозиторий и запускает **тот же** код проверки, что и в Docker (`model/evaluate.py`, `protocol.py`, `metrics.py`): те же люди, те же сценарии, те же метрики. Набор MovieLens 32M скачивается в память Colab.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**, затем «Среда выполнения» → «Выполнить все». В конце скачается архив `vae_<период>.zip` с отчётом – его нужно положить в `model/reports/`.

Время на T4 (оценка): загрузка набора и сценарии – около 2 минут, обучение четырёх вариантов – 20–40 минут, проверка – несколько минут. Вкладку не закрывать до конца.

In [ ]:
# проверочная часть: validation – выбор настроек; test – только на шаге 6, один раз
PERIOD = "validation"

In [ ]:
import torch

assert torch.cuda.is_available(), "нет GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

In [ ]:
# из /content: при повторном запуске в том же сеансе текущая папка – внутри удаляемого клона
%cd /content
!rm -rf /content/KinoCompass && git clone -q --depth 1 https://github.com/AlexTrav/KinoCompass.git /content/KinoCompass
!git -C /content/KinoCompass log --oneline -1
# те же версии, что в образе Docker (torch в Colab уже есть)
!pip install -q requests==2.34.2 pandas==3.1.0 pyarrow==26.0.1 scipy==1.18.1

In [ ]:
%cd /content/KinoCompass/model
!PYTHONPATH=/content/KinoCompass/data:/content/KinoCompass/research PERIOD={PERIOD} python -W ignore evaluate.py vae

In [ ]:
from google.colab import files

!cd /content/KinoCompass/model && zip -q -r /content/vae_{PERIOD}.zip reports/vae && ls -la reports/vae
files.download(f"/content/vae_{PERIOD}.zip")